In [29]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch as th
import sys
from pathlib import Path

work_root = Path.cwd().parent
if str(work_root) not in sys.path:
    sys.path.append(str(work_root))

import dataset.dataset

In [30]:
dataset.dataset.DATA_HUB['kaggle_house_train'] = (
    dataset.dataset.DATA_UIL + 'kaggle_house_pred_train.csv',
'585e9cc93e70b39160e7921475f9bcd7d31219ce'
)

dataset.dataset.DATA_HUB['kaggle_house_test'] = (
    dataset.dataset.DATA_UIL + 'kaggle_house_pred_test.csv',
'fa19780a7b011d9b009e8bff8e99922a8ee2eb90'
)

In [31]:
train_data = pd.read_csv(dataset.dataset.download('kaggle_house_train'))
test_data = pd.read_csv(dataset.dataset.download('kaggle_house_test'))

正在从http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv下载..\data\kaggle_house_pred_train.csv
正在从http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv下载..\data\kaggle_house_pred_test.csv


In [32]:
print(train_data.shape)
print(test_data.shape)
print(train_data.iloc[0].shape)
print(train_data.iloc[0:4,[0,1,2,-2,-1]])

(1460, 81)
(1459, 80)
(81,)
   Id  MSSubClass MSZoning SaleCondition  SalePrice
0   1          60       RL        Normal     208500
1   2          20       RL        Normal     181500
2   3          60       RL        Normal     223500
3   4          70       RL       Abnorml     140000


In [33]:
all_features = pd.concat((train_data.iloc[:,1:-1],test_data.iloc[:,1:]))

In [34]:
numeric_features = all_features.select_dtypes(include=["number"]).columns
all_features[numeric_features] = all_features[numeric_features].apply(lambda x: (x - x.mean())/(x.std()))

all_features[numeric_features] = all_features[numeric_features].fillna(0)

In [35]:
all_features = pd.get_dummies(all_features, dummy_na=True)
print(all_features.shape)

(2919, 330)


In [36]:
n_train = train_data.shape[0]
train_arr = all_features[:n_train].to_numpy(dtype=np.float32)
train_features = th.tensor(train_arr)

test_arr = all_features[n_train:].to_numpy(dtype=np.float32)
test_features = th.tensor(test_arr)

In [37]:
train_labels_arr = train_data.SalePrice.to_numpy(dtype=np.float32).reshape(-1,1)
train_labels = th.tensor(train_labels_arr)
print(train_labels[0:4])

tensor([[208500.],
        [181500.],
        [223500.],
        [140000.]])


In [38]:
import torch.nn as nn

In [ ]:
loss = nn.MSELoss
in_features = train_features.shape[1]

def get_net():
    net = nn.Sequential(nn.Linear(in_features,1))
    return net 

def log_rmse(features, net, labels):
    clipped_preds = torch.clamp(net(features), 1, float('inf'))
    rmse = th.sqrt(loss(torch.log(clipped_preds),th.log(labels)))
    return rmse.item()



In [40]:
def train(net, train_features, train_labels, test_features, test_labels, num_epochs, learning_rate, weight_decay, batch_size):
    train_ls,test_ls = [], []
    dataset = th.utils.data.TensorDataset(train_features, train_labels)
    dataload = th.utils.data.DataLoader(dataset,batch_size=batch_size,shuffle=True)

    optimizer = th.optim.Adam(net.parameters(),lr=learning_rate,weight_decay=weight_decay)

    for epoch in range(num_epochs):
        for X,y in dataload:
            optimizer.zero_grad()
            l = loss(net(X),y)
            l.backward()
            optimizer.step()

        train_ls.append(log_rmse(net,train_features,train_labels))
        if test_labels is not None:
            test_ls.append(log_rmse(net,test_features,test_labels))

    return train_ls, test_ls


In [41]:
import matplotlib as plt

def get_k_fold_data(K,i,X,y):
    assert K > 1
    folder_size = X.shape[0] // K
    X_train, y_train = None, None

    for j in range(K):
        idx = slice(j*folder_size,(j+1)*folder_size)
        X_part, y_part = X[idx, :], y[idx]
        if j ==  i:
            X_valid, y_valid = X_part, y_part
        elif X_train == None:
            X_train, y_train = X_part, y_part
        else:
            X_train = th.cat([X_train,X_part],0)
            y_train = th.cat([y_train,y_part],0)
    return X_train, y_train, X_valid, y_valid


def k_fold(K, X_train, y_train, num_epochs, learning_rate, weight_decay,batch_size):
    train_l_sum, valid_l_sum = 0,0
    for i in range(K):
        data = get_k_fold_data(K,i,X_train,y_train)
        net = get_net()

        train_ls, valid_ls = train(net, *data, num_epochs, learning_rate, weight_decay, batch_size)
        train_l_sum += train_ls[-1]
        valid_l_sum += valid_ls[-1]

        if i==0:
            plt.plot(list(range(1, num_epochs + 1)), [train_ls, valid_ls],
                        xlabel='epoch', ylabel='rmse', xlim=[1, num_epochs],
                        legend=['train', 'valid'], yscale='log'
                    )

        print(f'折{i + 1}，训练log rmse{float(train_ls[-1]):f}, 'f'验证log rmse{float(valid_ls[-1]):f}')
    return train_l_sum / K, valid_l_sum / K

In [42]:
k, num_epochs, lr, weight_decay, batch_size = 5, 100, 5, 0, 64
train_l, valid_l = k_fold(k, train_features, train_labels, num_epochs, lr,
weight_decay, batch_size)
print(f'{k}-折验证: 平均训练log rmse: {float(train_l):f}, '
f'平均验证log rmse: {float(valid_l):f}')

TypeError: 'Tensor' object is not callable